<a href="https://colab.research.google.com/github/igorfantucci/Aula-Automatica---GRUPO-5/blob/main/etapa-01-logica/07%20-%20Validade%20e%20Inferencia%20Logica%20na%20Seguranca%20de%20Processos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 07 - Notebook: Validade de Argumentos, Inferência Lógica e Prova de Consistência da Matriz de Segurança
## Processo: Planta Industrial de Produção de Biodiesel (Transesterificação em Batelada)

Neste notebook implementamos a classe **`ProvadorDedutivoFormal`** em Python para testar e comprovar matematicamente a validade lógica de argumentos de segurança operacional da **Planta de Produção de Biodiesel**. 

Exploramos:
1. **Método de Prova Exaustiva por Tabela-Verdade** ($2^n$ estados operacionais com identificação de linhas críticas e contraexemplos);
2. **Método de Verificação por Refutação (*Reductio ad Absurdum*)** fundamentado em solucionadores de satisfatibilidade booleana (*SAT Solvers* / IEC 61511);
3. **Bateria de Esquemas Canônicos de Inferência** (Modus Ponens, Modus Tollens, Silogismo Hipotético, Silogismo Disjuntivo, Resolução e Dilema Construtivo);
4. **Detecção e Rejeição Formal de Falácias Operacionais** (Afirmação do Consequente e Negação do Antecedente);
5. **Prova Lógica de Ausência de Falhas** na Matriz de Causa e Efeito (*Safety Matrix* / Sistema de Parada de Emergência ESD).

In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

import itertools
from typing import List, Dict, Callable, Any

class ProvadorDedutivoFormal:
    @staticmethod
    def verificar_argumento_tabela_verdade(
        variaveis: List[str],
        premissas: List[Callable[[Dict[str, bool]], bool]],
        conclusao: Callable[[Dict[str, bool]], bool]
    ) -> Dict[str, Any]:
        """
        Verifica a validade semântica do argumento: P1, P2, ..., Pk |- C
        Um argumento é válido se e somente se em toda linha onde todas as
        premissas são TRUE, a conclusão também é estritamente TRUE.
        """
        n = len(variaveis)
        total_estados = 2 ** n
        linhas_criticas = 0 # Linhas onde todas as premissas são verdadeiras
        linhas_validas = 0   # Linhas críticas onde a conclusão também é verdadeira
        contraexemplos = []
        
        for combo in itertools.product([False, True], repeat=n):
            env = dict(zip(variaveis, combo))
            premissas_satisfeitas = all(p(env) for p in premissas)
            
            if premissas_satisfeitas:
                linhas_criticas += 1
                if conclusao(env):
                    linhas_validas += 1
                else:
                    contraexemplos.append(env)
                    
        valido = (linhas_criticas > 0) and (linhas_criticas == linhas_validas)
        
        return {
            "Total Estados (2^n)": total_estados,
            "Estados com Premissas True": linhas_criticas,
            "Estados com Conclusão True": linhas_validas,
            "Válido": valido,
            "Resultado Semântico": "ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA)" if valido else "FALÁCIA / ARGUMENTO INVÁLIDO",
            "Contraexemplos": contraexemplos
        }

    @staticmethod
    def verificar_por_refutacao(
        variaveis: List[str],
        premissas: List[Callable[[Dict[str, bool]], bool]],
        conclusao: Callable[[Dict[str, bool]], bool]
    ) -> Dict[str, Any]:
        """
        Prova por Contradição / Refutação (SAT Solver approach):
        O argumento P1..Pk |- C é válido se e somente se o conjunto {P1, ..., Pk, NOT C}
        for INSATISFATÍVEL (CONTRADIÇÃO).
        """
        n = len(variaveis)
        modelos_refutacao = []
        
        for combo in itertools.product([False, True], repeat=n):
            env = dict(zip(variaveis, combo))
            if all(p(env) for p in premissas) and not conclusao(env):
                modelos_refutacao.append(env)
                
        is_insatisfativel = len(modelos_refutacao) == 0
        return {
            "Total Modelos": 2 ** n,
            "Modelos que Satisfazem {Premissas ∧ ¬C}": len(modelos_refutacao),
            "Refutação Bem-Sucedida": is_insatisfativel,
            "Conclusão": "PROVA POR CONTRADIÇÃO: ARGUMENTO VÁLIDO (INSATISFATÍVEL)" if is_insatisfativel else "REFUTAÇÃO FALHOU: CONTRADIÇÃO NÃO ENCONTRADA"
        }

print("[OK] Módulo ProvadorDedutivoFormal inicializado com sucesso!")

[OK] Módulo ProvadorDedutivoFormal inicializado com sucesso!


## 1. Bateria de Testes Formais: Regras Canônicas de Inferência e Falácias na Planta de Biodiesel

Testamos os 6 esquemas dedutivos fundamentais e 2 falácias operacionais aplicadas aos instrumentos e equipamentos da planta:
1. **Modus Ponens (MP):** Intertravamento Térmico do Reator R-200 ($t_{\text{alta}} \rightarrow \neg h_1, t_{\text{alta}} \vdash \neg h_1$);
2. **Modus Tollens (MT):** Diagnóstico de Falha na Bomba de Metanol P-102 ($b_{\text{met}} \rightarrow f_{\text{met}}, \neg f_{\text{met}} \vdash \neg b_{\text{met}}$);
3. **Silogismo Hipotético (SH):** Isolamento em Cadeia de Vapores no Setor 100 ($g_{\text{alm}} \rightarrow \neg v_{\text{met}}, \neg v_{\text{met}} \rightarrow \text{isola}_{s100} \vdash g_{\text{alm}} \rightarrow \text{isola}_{s100}$);
4. **Silogismo Disjuntivo (SD):** Salvaguarda de Resfriamento de Emergência no Reator ($r_1 \lor \text{trip}_{HT}, \neg r_1 \vdash \text{trip}_{HT}$);
5. **Resolução Proposicional (RES):** Fusão de Cláusulas de Intertravamento do Reator ($p_1 \lor m_{\text{falha}}, \neg p_1 \lor \text{alivio}_{psv} \vdash m_{\text{falha}} \lor \text{alivio}_{psv}$);
6. **Dilema Construtivo (DC):** Resposta a Múltiplas Contingências de Sobrepressão ou Transbordamento ($(p_1 \rightarrow \text{abre}_{psv}) \land (l_{\text{alto}} \rightarrow \text{corta}_{alim}), p_1 \lor l_{\text{alto}} \vdash \text{abre}_{psv} \lor \text{corta}_{alim}$);
7. **Afirmação do Consequente (Falácia):** Diagnóstico incorreto de sobrepressão a partir do fechamento de válvula ($p_1 \rightarrow \neg v_2, \neg v_2 \not\vdash p_1$);
8. **Negação do Antecedente (Falácia):** Habilitação incorreta do aquecedor baseada unicamente na ausência de sobrepressão ($p_1 \rightarrow \neg h_1, \neg p_1 \not\vdash h_1$).

In [2]:
# 1. Modus Ponens (MP): Trip Térmico no Reator R-200
vars_mp = ['t_alta', 'h1']
p1_mp = lambda env: (not env['t_alta']) or (not env['h1']) # t_alta -> ~h1
p2_mp = lambda env: env['t_alta']                          # Fato: t_alta
c_mp  = lambda env: not env['h1']                          # Conclusao: ~h1
res_mp = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_mp, [p1_mp, p2_mp], c_mp)

# 2. Modus Tollens (MT): Diagnóstico de Bomba de Metanol P-102
vars_mt = ['b_met', 'f_met']
p1_mt = lambda env: (not env['b_met']) or env['f_met']     # b_met -> f_met
p2_mt = lambda env: not env['f_met']                       # Fato: not f_met
c_mt  = lambda env: not env['b_met']                       # Conclusao: not b_met
res_mt = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_mt, [p1_mt, p2_mt], c_mt)

# 3. Silogismo Hipotético (SH): Isolamento Setor 100
vars_sh = ['g_alm', 'v_met', 'isola_s100']
p1_sh = lambda env: (not env['g_alm']) or (not env['v_met'])
p2_sh = lambda env: env['v_met'] or env['isola_s100']       # ~v_met -> isola_s100
c_sh  = lambda env: (not env['g_alm']) or env['isola_s100'] # g_alm -> isola_s100
res_sh = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_sh, [p1_sh, p2_sh], c_sh)

# 4. Silogismo Disjuntivo (SD): Resfriamento de Emergência
vars_sd = ['r1', 'trip_ht']
p1_sd = lambda env: env['r1'] or env['trip_ht']
p2_sd = lambda env: not env['r1']
c_sd  = lambda env: env['trip_ht']
res_sd = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_sd, [p1_sd, p2_sd], c_sd)

# 5. Resolução Proposicional (RES): Intertravamento do Reator
vars_res = ['p1', 'falha_agit', 'alivio_psv']
p1_res = lambda env: env['p1'] or env['falha_agit']
p2_res = lambda env: (not env['p1']) or env['alivio_psv']
c_res  = lambda env: env['falha_agit'] or env['alivio_psv']
res_res = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_res, [p1_res, p2_res], c_res)

# 6. Dilema Construtivo (DC): Mitigação Múltipla R-200
vars_dc = ['p1', 'abre_psv', 'l_alto', 'corta_alim']
p1_dc = lambda env: ((not env['p1']) or env['abre_psv']) and ((not env['l_alto']) or env['corta_alim'])
p2_dc = lambda env: env['p1'] or env['l_alto']
c_dc  = lambda env: env['abre_psv'] or env['corta_alim']
res_dc = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_dc, [p1_dc, p2_dc], c_dc)

# 7. Falácia da Afirmação do Consequente (INVÁLIDO)
vars_fal1 = ['p1', 'v_met']
p1_fal1 = lambda env: (not env['p1']) or (not env['v_met'])
p2_fal1 = lambda env: not env['v_met']
c_fal1  = lambda env: env['p1']
res_fal1 = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_fal1, [p1_fal1, p2_fal1], c_fal1)

# 8. Falácia da Negação do Antecedente (INVÁLIDO)
vars_fal2 = ['t_alta', 'h1']
p1_fal2 = lambda env: (not env['t_alta']) or (not env['h1'])
p2_fal2 = lambda env: not env['t_alta']
c_fal2  = lambda env: env['h1']
res_fal2 = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(vars_fal2, [p1_fal2, p2_fal2], c_fal2)

relatorio_testes = [
    {"Esquema Lógico": "Modus Ponens (MP)", "Setor / Equipamento": "Reator R-200 (HT-201)", "Resultado Semântico": res_mp["Resultado Semântico"], "Válido": res_mp["Válido"]},
    {"Esquema Lógico": "Modus Tollens (MT)", "Setor / Equipamento": "Metanol Setor 100 (P-102)", "Resultado Semântico": res_mt["Resultado Semântico"], "Válido": res_mt["Válido"]},
    {"Esquema Lógico": "Silogismo Hipotético (SH)", "Setor / Equipamento": "Isolamento Setor 100", "Resultado Semântico": res_sh["Resultado Semântico"], "Válido": res_sh["Válido"]},
    {"Esquema Lógico": "Silogismo Disjuntivo (SD)", "Setor / Equipamento": "Resfriamento Emergência R-200", "Resultado Semântico": res_sd["Resultado Semântico"], "Válido": res_sd["Válido"]},
    {"Esquema Lógico": "Resolução Proposicional (RES)", "Setor / Equipamento": "Intertravamento Reator R-200", "Resultado Semântico": res_res["Resultado Semântico"], "Válido": res_res["Válido"]},
    {"Esquema Lógico": "Dilema Construtivo (DC)", "Setor / Equipamento": "Mitigação Múltipla R-200", "Resultado Semântico": res_dc["Resultado Semântico"], "Válido": res_dc["Válido"]},
    {"Esquema Lógico": "Afirmação Consequente (Falácia)", "Setor / Equipamento": "Diagnóstico de Sobrepressão", "Resultado Semântico": res_fal1["Resultado Semântico"], "Válido": res_fal1["Válido"]},
    {"Esquema Lógico": "Negação Antecedente (Falácia)", "Setor / Equipamento": "Habilitação Indevida Aquecedor", "Resultado Semântico": res_fal2["Resultado Semântico"], "Válido": res_fal2["Válido"]}
]

print("=== RELATÓRIO 1: VERIFICAÇÃO FORMAL DE REGRAS DE INFERÊNCIA E FALÁCIAS ===")
print(formatar_tabela(relatorio_testes))

assert res_mp["Válido"] is True
assert res_mt["Válido"] is True
assert res_sh["Válido"] is True
assert res_sd["Válido"] is True
assert res_res["Válido"] is True
assert res_dc["Válido"] is True
assert res_fal1["Válido"] is False
assert res_fal2["Válido"] is False
print("\n[OK] Todos os testes de inferência dedutiva e detecção de falácias passaram com 100% de sucesso!")

=== RELATÓRIO 1: VERIFICAÇÃO FORMAL DE REGRAS DE INFERÊNCIA E FALÁCIAS ===
Esquema Lógico                  | Setor / Equipamento            | Resultado Semântico                     | Válido
--------------------------------+--------------------------------+-----------------------------------------+-------
Modus Ponens (MP)               | Reator R-200 (HT-201)          | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True  
Modus Tollens (MT)              | Metanol Setor 100 (P-102)      | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True  
Silogismo Hipotético (SH)       | Isolamento Setor 100           | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True  
Silogismo Disjuntivo (SD)       | Resfriamento Emergência R-200  | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True  
Resolução Proposicional (RES)   | Intertravamento Reator R-200   | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True  
Dilema Construtivo (DC)         | Mitigação Múltipla R-200       | ARGUMENTO VÁLIDO (TEOREMA DE SEGURANÇA) | True

## 2. Matriz de Causa e Efeito (C&E / Safety Matrix) e Prova Lógica de Ausência de Falhas

Nesta seção modelamos as $11$ variáveis proposicionais de segurança da planta de produção de biodiesel e provamos dois teoremas fundamentais de segurança funcional por **Tabela-Verdade Exaustiva ($2^{11} = 2048$ estados)** e por **Refutação / SAT Solver**:

* **Teorema 1 (Parada Global de Emergência):** Quando o botão de emergência $e_1$ for acionado, $100\%$ dos atuadores críticos de todos os setores são forçados para a posição segura:
$$\mathcal{M}_{\text{ESD}} \vdash e_1 \rightarrow (\neg h_1 \land \neg v_2 \land \neg b_{\text{met}} \land \neg v_3 \land \neg b_1)$$

* **Teorema 2 (Proteção Térmica e de Pressão do Reator):** Sob sobrepressão ($p_1$) ou sobretemperatura ($t_{\text{alta}}$), a alimentação química ($v_2$) e a fonte de calor ($h_1$) são infalivelmente neutralizadas:
$$\mathcal{M}_{\text{ESD}} \vdash (p_1 \lor t_{\text{alta}}) \rightarrow (\neg h_1 \land \neg v_2)$$

In [3]:
# Variáveis da Malha de Segurança:
# Entradas: p1, t_alta, l_alto, g_alm, e1, r1
# Saídas:   h1, v2, b_met, v3, b1
vars_matriz = ['p1', 't_alta', 'l_alto', 'g_alm', 'e1', 'r1', 'h1', 'v2', 'b_met', 'v3', 'b1']

# Regras de Intertravamento da Matriz de Causa e Efeito (C&E Matrix):
p_matriz_h1    = lambda env: (not (env['p1'] or env['t_alta'] or env['e1'] or not env['r1'])) or (not env['h1'])
p_matriz_v2    = lambda env: (not (env['p1'] or env['t_alta'] or env['l_alto'] or env['g_alm'] or env['e1'])) or (not env['v2'])
p_matriz_b_met = lambda env: (not (env['g_alm'] or env['e1'])) or (not env['b_met'])
p_matriz_v3    = lambda env: (not env['e1']) or (not env['v3'])
p_matriz_b1    = lambda env: (not env['e1']) or (not env['b1'])

premissas_esd = [p_matriz_h1, p_matriz_v2, p_matriz_b_met, p_matriz_v3, p_matriz_b1]

# Teorema 1: Parada Total de Emergência (e1 -> FailSafe Global)
conclusao_esd_total = lambda env: (not env['e1']) or (
    (not env['h1']) and (not env['v2']) and (not env['b_met']) and (not env['v3']) and (not env['b1'])
)

res_esd_total = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(
    vars_matriz, premissas_esd, conclusao_esd_total
)
ref_esd_total = ProvadorDedutivoFormal.verificar_por_refutacao(
    vars_matriz, premissas_esd, conclusao_esd_total
)

# Teorema 2: Trip Térmico e de Pressão do Reator ((p1 v t_alta) -> ~h1 ∧ ~v2)
conclusao_reator_safe = lambda env: (not (env['p1'] or env['t_alta'])) or (
    (not env['h1']) and (not env['v2'])
)

res_reator_safe = ProvadorDedutivoFormal.verificar_argumento_tabela_verdade(
    vars_matriz, premissas_esd, conclusao_reator_safe
)
ref_reator_safe = ProvadorDedutivoFormal.verificar_por_refutacao(
    vars_matriz, premissas_esd, conclusao_reator_safe
)

relatorio_esd = [
    {"Teorema de Segurança": "T1: Parada Total de Emergência (e1 -> FailSafe Global)", "Total Estados": res_esd_total["Total Estados (2^n)"], "Refutação": ref_esd_total["Refutação Bem-Sucedida"], "Resultado": "PROVA DE AUSÊNCIA DE FALHA (100% SEGURO)" if res_esd_total["Válido"] else "FALHA DETECTADA"},
    {"Teorema de Segurança": "T2: Trip Térmico/Pressão Reator ((p1 v t_alta) -> ~h1 ∧ ~v2)", "Total Estados": res_reator_safe["Total Estados (2^n)"], "Refutação": ref_reator_safe["Refutação Bem-Sucedida"], "Resultado": "PROVA DE AUSÊNCIA DE FALHA (100% SEGURO)" if res_reator_safe["Válido"] else "FALHA DETECTADA"}
]

print("=== RELATÓRIO 2: PROVA FORMAL DE AUSÊNCIA DE FALHAS NA MATRIZ DE SEGURANÇA (ESD) ===")
print(formatar_tabela(relatorio_esd))

assert res_esd_total["Válido"] is True
assert res_reator_safe["Válido"] is True
assert ref_esd_total["Refutação Bem-Sucedida"] is True
assert ref_reator_safe["Refutação Bem-Sucedida"] is True

print("\n[SUCESSO] Prova Formal Concluída: 0 contraexemplos encontrados em 2048 combinações de estados operacionais!")
print("[OK] A matriz de segurança do complexo de biodiesel é formalmente consistente e completa.")

=== RELATÓRIO 2: PROVA FORMAL DE AUSÊNCIA DE FALHAS NA MATRIZ DE SEGURANÇA (ESD) ===
Teorema de Segurança                                         | Total Estados | Refutação | Resultado                               
-------------------------------------------------------------+---------------+-----------+-----------------------------------------
T1: Parada Total de Emergência (e1 -> FailSafe Global)       | 2048          | True      | PROVA DE AUSÊNCIA DE FALHA (100% SEGURO)
T2: Trip Térmico/Pressão Reator ((p1 v t_alta) -> ~h1 ∧ ~v2) | 2048          | True      | PROVA DE AUSÊNCIA DE FALHA (100% SEGURO)

[SUCESSO] Prova Formal Concluída: 0 contraexemplos encontrados em 2048 combinações de estados operacionais!
[OK] A matriz de segurança do complexo de biodiesel é formalmente consistente e completa.
